# AD xQTL loci table

Reproducible pipeline to generate the unified AD xQTL loci summary Excel table and deploy the Shiny explorer.

**To add new data or cell types:** edit only the `CONFIG` cell below. All downstream steps are fully driven by those definitions.


## CONFIG

**Edit this cell only** when updating input files, adding new cell types/QTL modalities, or changing filter thresholds.


In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────
set.seed(42)

# ── Input/output paths ────────────────────────────────────────────────────
input_dir        <- "/restricted/projectnb/xqtl/jaempawi/xqtl/AD_xQTL_loci"
flatten_file     <- file.path(input_dir, "xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz")
context_meta_file <- file.path(input_dir, "context_meta.tsv")
gene_ref_file    <- file.path(input_dir, "Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list")
output_file      <- file.path(input_dir, "AD_loci_xQTL_table.xlsx")
shiny_app_dir    <- file.path(input_dir, "shiny_app")

# ── Variant filter thresholds ─────────────────────────────────────────────
# Variants passing any of these criteria are included in the table
INCLUSION_PROB_THRESHOLD <- 0.1   # max_variant_inclusion_probability >= this
CV2F_RANK_THRESHOLD      <- 5     # cV2F_rank <= this

# GWAS significance thresholds
PVAL_GENOME_WIDE <- 5e-8
PVAL_SUGGESTIVE  <- 1e-6

# Max ordered context columns in Gene Locus sheet
MAX_CTX_GENE_LOCUS <- 14L

# ── Cell-type order and display labels ───────────────────────────────────
# To add a new cell type: append to ct_order, add a display label,
# and add its QTL modalities to ct_qtl_order below.
ct_order <- c("Brain", "Exc", "Inh", "Oli", "OPC", "Ast", "Mic", "Immune (bulk)")

ct_display_label <- c(
  "Brain"         = "Brain xQTL",
  "Exc"           = "Exc xQTL",
  "Inh"           = "Inh xQTL",
  "Oli"           = "Oli xQTL",
  "OPC"           = "OPC xQTL",
  "Ast"           = "Ast xQTL",
  "Mic"           = "Microglia xQTL",
  "Immune (bulk)" = "Bulk Immune xQTL"
)

# ── QTL modalities per cell type ─────────────────────────────────────────
# To add a new QTL modality to an existing cell type: append the QTL label here.
# To add a new QTL modality type entirely: also add its column map in ct_col_map (STEP 8b).
ct_qtl_order <- list(
  Brain           = c("eQTL", "pQTL", "gpQTL", "sQTL", "p_sQTL", "u_sQTL", "haQTL", "mQTL"),
  Exc             = c("eQTL", "sQTL"),
  Inh             = c("eQTL", "sQTL"),
  Oli             = c("eQTL", "sQTL"),
  OPC             = c("eQTL", "sQTL"),
  Ast             = c("eQTL", "sQTL", "caQTL"),
  Mic             = c("eQTL", "sQTL", "caQTL"),
  `Immune (bulk)` = c("eQTL")
)

# ── Non-linear effect column counts per cell type (for header construction) 
# Brain/Exc have 8 (APOE4 + sex interactions); Immune (bulk) has 6 (sex only)
ct_non_linear_cols <- c(
  Brain = 8L, Exc = 8L,
  Inh = 4L, Oli = 6L, OPC = 4L,
  Ast = 4L, Mic = 4L, `Immune (bulk)` = 6L
)


## Setup: libraries & utils

In [ ]:
library(data.table)
library(stringr)
library(tidyverse)
library(openxlsx)
suppressWarnings(source("/restricted/projectnb/xqtl/jaempawi/xqtl/Figure_6/gene_prio_utils.R"))


## Validate inputs

In [ ]:
# Check all required input files exist before running
required_files <- c(flatten_file, context_meta_file, gene_ref_file)
missing <- required_files[!file.exists(required_files)]
if (length(missing) > 0) {
  stop("Missing input files:\n", paste(" -", missing, collapse="\n"))
}
cat("All input files found.\n")
cat("  flatten_file    :", flatten_file, "\n")
cat("  context_meta    :", context_meta_file, "\n")
cat("  gene_ref        :", gene_ref_file, "\n")
cat("  output          :", output_file, "\n")


## STEP 1–2: Load flatten & set up contexts

In [ ]:
# ── Load flatten ─────────────────────────────────────────────────────────
cat("Loading flatten...\n")
res_adx <- fread(flatten_file)
cat("  ", nrow(res_adx), "rows\n")
cat("  Unique genes:", uniqueN(res_adx$gene_name), "\n")
cat("  Methods:\n"); print(table(res_adx$Method))
cat("  cTWAS_signif TRUE:", sum(res_adx$cTWAS_signif, na.rm=TRUE), "\n")

# ── Derive context_broad2 and qtl_type if not already present ───────────
if (!"context_broad2" %in% names(res_adx)) {
  res_adx[, context_broad2 := ifelse(
    context_broad %in% c('bulk_monocyte_eQTL','bulk_macrophage_eQTL','bulk_microglia_eQTL'),
    'Immune (bulk)',
    ifelse(str_detect(context_broad, 'bulk_brain'), 'Brain',
           str_remove(context_broad, '_eQTL|_snATAC|_brain|_sQTL')))]
}
if (!"qtl_type" %in% names(res_adx)) {
  res_adx[, qtl_type := str_extract(context_broad, '(u_|p_)?[a-z]+QTL')]
  res_adx[str_detect(context_broad, 'snATAC'), qtl_type := 'caQTL']
}

# ── Backfill optional columns absent from older flatten versions ─────────
if (!"MR_signif"   %in% names(res_adx)) res_adx[, MR_signif   := FALSE]
if (!"TWAS_signif" %in% names(res_adx)) res_adx[, TWAS_signif := FALSE]
if (!"cos_npc"     %in% names(res_adx)) res_adx[, cos_npc     := NA_real_]

# ── Cap Inf TWAS z-scores at the observed max finite value ───────────────
twas_finite <- res_adx[!is.infinite(twas_z) & !is.na(twas_z)]$twas_z
twas_max    <- max(abs(twas_finite), na.rm = TRUE)
res_adx[twas_z ==  Inf, twas_z :=  twas_max]
res_adx[twas_z == -Inf, twas_z := -twas_max]


## STEP 3: SummarizeTable

In [ ]:
cat("\nRunning SummarizeTable...\n")
res_adx <- suppressWarnings(
  SummarizeTable(res_adx, group.by = 'context_short')
)
cat("  Done.", nrow(res_adx), "rows\n")


## STEP 4: WideTable & variant filter

In [ ]:
# Save long-format before pivoting (needed for ordered contexts in STEP 8a)
res_adx_long <- res_adx

cat("\nRunning WideTable...\n")
res_adxub <- suppressWarnings(
  WideTable(res_adx, split.by = c('context_broad2', 'qtl_type'))
)
cat("  Done.", nrow(res_adxub), "rows\n")
cat("  Unique variant-gene pairs:", uniqueN(res_adxub[, .(locus_index, variant_ID, gene_name)]), "\n")

# ── Variant filter (thresholds set in CONFIG) ───────────────────────────
cat("\nApplying variant filter...\n")
res_adxub[, top_variants := (
  (max_variant_inclusion_probability >= INCLUSION_PROB_THRESHOLD) | cV2F_rank <= CV2F_RANK_THRESHOLD) |
  ((max_variant_inclusion_probability_rank == 1 |
    variant_rank_xqtl == 1 |
    (rank(pval) == 1 & !is.na(pval))
  )), by = 'locus_index']

selected_variants <- res_adxub[(top_variants) & !is.na(locus_index) & variant_ID != '']
cat("  Selected", nrow(selected_variants), "rows\n")
cat("  Unique loci:", uniqueN(selected_variants$locus_index), "\n")
cat("  Unique variant-gene pairs:", uniqueN(selected_variants[, .(locus_index, variant_ID, gene_name)]), "\n")


## STEP 5: Derived columns

In [ ]:
selected_variants[, gwas_significance := ifelse(
  min_pval < PVAL_GENOME_WIDE, "genome wide",
  ifelse(min_pval < PVAL_SUGGESTIVE, "suggestive", "ns"))]

selected_variants[, top_confidence := str_replace(
  str_extract(xQTL_effects, 'CL?[1-6]'), '^C([1-6])$', 'CL\\1')]

cat("\n  Significance:\n");  print(table(selected_variants$gwas_significance, useNA="ifany"))
cat("\n  Confidence:\n");    print(table(selected_variants$top_confidence,    useNA="ifany"))

variant_gene <- selected_variants
cat("\n  TWAS_signif_gene:\n");  print(table(variant_gene$TWAS_signif_gene,  useNA="ifany"))
cat("\n  cTWAS_signif_gene:\n"); print(table(variant_gene$cTWAS_signif_gene, useNA="ifany"))


## STEP 6: Locus/variant columns

In [ ]:
cat("\nBuilding locus/variant columns...\n")
locus_variant_cols <- selected_variants %>%
  distinct(locus_index, variant_ID, .keep_all = TRUE) %>%
  transmute(
    locus_index                          = locus_index,
    ADlocus                              = ADlocus,
    ADlocusID                            = ADlocusID,
    `# variants in this locus`          = n.variants,
    `# with inclusion score >01`        = n.variants_0.1,
    variant_ID                           = variant_ID,
    Rsid                                 = SNP,
    Chr                                  = chr,
    Pos                                  = pos,
    `Effect allele`                      = effect_allele,
    `cV2F score`                         = cV2F,
    `maximum inclusion score`            = max_variant_inclusion_probability,
    `maximum inclusion score method`     = max_variant_inclusion_probability_method,
    `cV2F rank`                          = cV2F_rank,
    `Maximum zscore`                     = max_zscore,
    `Min p-value`                        = min_pval,
    Significance                         = gwas_significance,
    `log10(pval)`                        = -log10(min_pval),
    `GWAS associated to this variant`    = gwas_sources,
    `GWAS Sources`                       = gwas_sources_effects,
    `Detected methods`                   = GWAS_methods
  )
cat("  locus_variant_cols rows:", nrow(locus_variant_cols), "\n")


## STEP 7: Gene-level summary

In [ ]:
cat("\nBuilding gene-level summary columns...\n")
gene_summary <- variant_gene %>%
  group_by(locus_index, variant_ID, gene_name) %>%
  summarise(
    `xQTL target gene`                      = first(gene_name),
    `gene ID`                               = first(gene_ID),
    tss                                     = first(tss),
    tes                                     = first(tes),
    `Distance from TSS`                     = first(distance_from_tss),
    `Distance from TES`                     = first(distance_from_tes),
    `Maximum TWAS zscore`                   = first(twas_z_gene_max),
    `Maximum TWAS context`                  = first(twas_z_gene_max_context),
    `TWAS significant`                      = first(TWAS_signif_gene),
    `MR significant`                        = first(MR_signif_gene),
    `cTWAS significant`                     = first(cTWAS_signif_gene),
    `Maximum inclusion score`               = suppressWarnings(
                                               max(variant_inclusion_probability_xqtl, na.rm=TRUE)) %>%
                                               {ifelse(is.infinite(.), NA_real_, .)},
    `Variant rank`                          = first(variant_rank_xqtl),
    Context                                 = first(context_xqtl),
    effect                                  = first(xQTL_effect),
    `# variants within the loci`           = first(n.variant_xqtl),
    `# molecular contexts (datasets)`      = first(n_contexts),
    `max confidence level` = case_when(
      is.na(first(top_confidence))               ~ NA_character_,
      grepl("^CL[1-6]$", first(top_confidence)) ~ first(top_confidence),
      grepl("^C[1-6]$",  first(top_confidence)) ~ paste0("CL", substr(first(top_confidence),2,2)),
      grepl("^[1-6]$",   first(top_confidence)) ~ paste0("CL", first(top_confidence)),
      TRUE                                       ~ NA_character_
    ),
    datasets                                = first(xQTL_contexts),
    Methods                                 = first(xQTL_methods),
    `Credible set coverage`                 = first(coverage_xqtl),
    `# trans genes`                         = first(n_trans_genes),
    `# trans contexts`                      = first(n_trans_contexts),
    `# genes`                               = first(n.mv_genes),
    genes                                   = first(mv_genes),
    `QR contexts`                           = first(qr_contexts),
    `Top QR context`                        = first(top_qr_context),
    `Top QR classification`                 = first(top_qr_classification),
    `Top QR adjusted P-value (bonferroni)`  = first(top_qr_p_bonferroni_adj),
    `Interactions with APOE4`               = first(apoe4_interactions),
    `min p-value for APOE4 interaction`     = first(top_apoe4_p_value),
    `Interactions with Sex (M)`             = first(msex_interactions),
    `min p-value for Sex interaction`       = first(top_msex_p_value),
    .groups = "drop"
  )
cat("  gene_summary rows:", nrow(gene_summary), "\n")
cat("  Confidence:\n");        print(table(gene_summary$`max confidence level`, useNA="ifany"))
cat("  TWAS significant:\n");  print(table(gene_summary$`TWAS significant`,     useNA="ifany"))
cat("  cTWAS significant:\n"); print(table(gene_summary$`cTWAS significant`,    useNA="ifany"))


## STEP 8a: Ordered contexts

In [ ]:
cat("\nBuilding ordered contexts...\n")
selected_vg <- variant_gene %>% distinct(locus_index, variant_ID, gene_name)
ordered_contexts <- res_adx_long %>%
  inner_join(selected_vg, by = c("locus_index", "variant_ID", "gene_name")) %>%
  filter(!is.na(confidence_lvl), context_broad2 != "GWAS", !is.na(xQTL_effect)) %>%
  distinct(locus_index, variant_ID, gene_name, context_broad2, qtl_type,
           confidence_lvl, n_study, context_short, xQTL_effect) %>%
  mutate(
    cl_num        = as.numeric(str_extract(confidence_lvl, "\\d+")),
    effect_suffix = str_remove(xQTL_effect, fixed(context_short)),
    abbrev_ctx    = case_when(
      context_broad2 == "Immune (bulk)" ~ "bulk",
      TRUE                              ~ str_extract(context_short, "^\\S+")
    ),
    label = paste0(abbrev_ctx, " ", qtl_type, effect_suffix)
  ) %>%
  group_by(locus_index, variant_ID, gene_name, context_broad2, qtl_type) %>%
  slice_min(cl_num, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(locus_index, variant_ID, gene_name, cl_num) %>%
  group_by(locus_index, variant_ID, gene_name) %>%
  summarise(ordered_ctx = list(label), .groups = "drop")

spread_ordered_ctx <- function(ctx_df, n_cols) {
  ctx_df %>%
    mutate(ctx_padded = map(ordered_ctx, ~ c(.x, rep(NA_character_, n_cols))[1:n_cols])) %>%
    unnest_wider(ctx_padded, names_sep = "_") %>%
    select(-ordered_ctx)
}

# MAX_CTX_GENE_LOCUS set in CONFIG
ordered_contexts_wide_gl <- spread_ordered_ctx(ordered_contexts, MAX_CTX_GENE_LOCUS)
cat("  ordered_contexts rows:", nrow(ordered_contexts_wide_gl), "\n")


## STEP 8b: QTL column map

Defines which metrics appear in the Excel columns for each QTL modality.
To add a new modality, add an entry here and register it in `ct_qtl_order` in CONFIG.


In [ ]:
# ── QTL column map ──────────────────────────────────────────────────────
# Keys are WideTable column prefixes; values are Excel display names.
# To add a new QTL type: add a named list entry below, then add the type
# to ct_qtl_order in CONFIG.
ct_col_map <- list(
  eQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "max_finemap_pip"            = "Fine mapping PIP",
    "max_finemap_effect"         = "Fine mapping effect",
    "max_coloc_vcp"              = "Colocalization VCP",
    "max_coloc_effect"           = "Colocalization effect",
    "max_multicontext_pip"       = "Multi-context fine mapping PIP",
    "max_multicontext_effect"    = "Multi-context fine mapping effect",
    "max_TWAS_Z"                 = "TWAS zscore",
    "max_MR"                     = "MR significant",
    "max_QR_padj"                = "QR bonferroni adj Pval",
    "max_APOE4_interaction_pval" = "APOE4 interaction Pvalue",
    "max_msex_interaction_pval"  = "Sex interaction Pvalue",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  pQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "max_finemap_pip"            = "Fine mapping PIP",
    "max_finemap_effect"         = "Fine mapping effect",
    "max_coloc_vcp"              = "Colocalization VCP",
    "max_coloc_effect"           = "Colocalization effect",
    "max_multicontext_pip"       = "Multi-context fine mapping PIP",
    "max_multicontext_effect"    = "Multi-context fine mapping effect",
    "max_TWAS_Z"                 = "TWAS zscore",
    "max_MR"                     = "MR significant",
    "max_QR_padj"                = "QR bonferroni adj Pval",
    "max_APOE4_interaction_pval" = "APOE4 interaction Pvalue",
    "max_msex_interaction_pval"  = "Sex interaction Pvalue",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  gpQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "max_finemap_pip"            = "Fine mapping PIP",
    "max_finemap_effect"         = "Fine mapping effect",
    "max_coloc_vcp"              = "Colocalization VCP",
    "max_coloc_effect"           = "Colocalization effect",
    "max_TWAS_Z"                 = "TWAS zscore",
    "max_MR"                     = "MR significant",
    "glyco_changes"              = "Glyco changes",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  sQTL = c(
    "max_dataset"                = "Dataset",
    "max_njunctions"             = "# splice sites diff spliced (FDR<005)",
    "max_junctions"              = "Splice sites",
    "max_junctions_type"         = "Splice types",
    "max_sn_splice_pval"         = "pvalue",
    "max_sn_splice_beta"         = "beta",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  p_sQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "max_finemap_pip"            = "Fine mapping PIP",
    "max_finemap_effect"         = "Fine mapping effect",
    "max_coloc_vcp"              = "Colocalization VCP",
    "max_coloc_effect"           = "Colocalization effect",
    "max_TWAS_Z"                 = "TWAS zscore",
    "max_MR"                     = "MR significant",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  u_sQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "max_finemap_pip"            = "Fine mapping PIP",
    "max_finemap_effect"         = "Fine mapping effect",
    "max_coloc_vcp"              = "Colocalization VCP",
    "max_coloc_effect"           = "Colocalization effect",
    "max_TWAS_Z"                 = "TWAS zscore",
    "max_MR"                     = "MR significant",
    "max_njunctions"             = "# junctions displiced",
    "max_junctions"              = "Junctions",
    "max_junctions_type"         = "Junctions type",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  haQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "top_epi_mark"               = "Epigenetics mark",
    "top_epi_mark_position"      = "Epigenetics mark position",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  mQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "top_epi_mark"               = "Epigenetics mark",
    "top_epi_mark_position"      = "Epigenetics mark position",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  ),
  caQTL = c(
    "max_inclusion_probability"  = "Maximum inclusion score",
    "max_dataset"                = "Dataset",
    "n_datasets"                 = "#total datasets",
    "datasets"                   = "Datasets or molecular contexts"
  )
)


## STEP 8c: Build wide block lists

In [ ]:
cat("\nBuilding cell-type specific columns...\n")
gl_wide_list <- list()
ct_wide_list <- list()

base_data <- variant_gene %>%
  distinct(locus_index, variant_ID, gene_name, .keep_all = TRUE)

# Auto-detect WideTable column suffix (.x suffix appears when cols were duplicated)
test_col   <- names(base_data)[grep("max_inclusion_probability\\.Brain\\.eQTL", names(base_data))][1]
wide_suffix <- if (!is.na(test_col) && grepl("\\.x$", test_col)) ".x" else ""
cat("  WideTable column suffix:", ifelse(wide_suffix == "", "(none)", wide_suffix), "\n")

# ct_order and ct_qtl_order are defined in CONFIG
for (ct in ct_order) {
  for (qt in ct_qtl_order[[ct]]) {
    block_name <- paste0(ct, ".", qt)
    qt_map     <- ct_col_map[[qt]]
    if (is.null(qt_map)) next
    new_cols <- list()
    for (metric in names(qt_map)) {
      wide_col  <- paste0(metric, ".", ct, ".", qt, wide_suffix)
      disp_name <- paste0(ct, ".", qt, ".", qt_map[[metric]])
      new_cols[[disp_name]] <- if (wide_col %in% names(base_data)) base_data[[wide_col]] else rep(NA, nrow(base_data))
    }
    block <- bind_cols(
      base_data %>% select(locus_index, variant_ID, gene_name),
      as_tibble(new_cols)
    )
    block_filtered <- block %>% filter(if_any(all_of(names(new_cols)), ~ !is.na(.x)))
    if (nrow(block_filtered) > 0) {
      gl_wide_list[[block_name]] <- block_filtered
      ct_wide_list[[block_name]] <- block_filtered
    }
  }
}
cat("  gl_wide_list blocks:", length(gl_wide_list), "\n")
cat("  Blocks:", paste(names(gl_wide_list), collapse=", "), "\n")


## STEP 8d: Assemble Gene Locus table

In [ ]:
cat("\nAssembling Gene Locus table and per-cell-type sheet data...\n")

gl_table <- gene_summary %>%
  left_join(locus_variant_cols,        by = c("locus_index", "variant_ID")) %>%
  left_join(ordered_contexts_wide_gl,  by = c("locus_index", "variant_ID", "gene_name"))
for (block_name in names(gl_wide_list)) {
  gl_table <- gl_table %>%
    left_join(gl_wide_list[[block_name]], by = c("locus_index", "variant_ID", "gene_name"))
}

# Add variants with no gene associations as stub rows
variants_no_genes <- locus_variant_cols %>%
  anti_join(gl_table %>% distinct(locus_index, variant_ID),
            by = c("locus_index", "variant_ID"))
if (nrow(variants_no_genes) > 0) {
  cat("  Adding", nrow(variants_no_genes), "variants with no gene associations\n")
  gl_table <- bind_rows(gl_table, variants_no_genes %>% mutate(gene_name = NA_character_))
}
gl_table <- gl_table %>% arrange(locus_index, variant_ID, gene_name)
cat("  Gene Locus table:", nrow(gl_table), "rows x", ncol(gl_table), "cols\n")

# Per-cell-type sheet data
ct_sheet_data <- list()
for (ct in ct_order) {
  sheet_tbl <- gene_summary
  for (qt in ct_qtl_order[[ct]]) {
    block_name <- paste0(ct, ".", qt)
    if (block_name %in% names(ct_wide_list))
      sheet_tbl <- sheet_tbl %>%
        left_join(ct_wide_list[[block_name]], by = c("locus_index", "variant_ID", "gene_name"))
  }
  ct_prefixed_cols <- grep(
    paste0("^", gsub("\\(", "\\\\(", gsub("\\)", "\\\\)", ct)), "\\."),
    names(sheet_tbl), value = TRUE
  )
  sheet_tbl <- if (length(ct_prefixed_cols) > 0)
    sheet_tbl %>% filter(if_any(all_of(ct_prefixed_cols), ~ !is.na(.x)))
  else sheet_tbl[0, ]
  ct_sheet_data[[ct]] <- sheet_tbl
  cat("  Sheet '", ct, "':", nrow(sheet_tbl), "rows\n")
}

# Per-sheet ordered contexts
ct_ordered_ctx <- list()
for (ct in ct_order) {
  ct_ctx <- ordered_contexts %>%
    semi_join(ct_sheet_data[[ct]], by = c("locus_index", "variant_ID", "gene_name"))
  if (nrow(ct_ctx) == 0) {
    ct_ordered_ctx[[ct]] <- tibble(locus_index=integer(), variant_ID=character(), gene_name=character())
    next
  }
  max_ctx <- max(ct_ctx %>% mutate(n=lengths(ordered_ctx)) %>% pull(n), na.rm=TRUE)
  ct_ordered_ctx[[ct]] <- spread_ordered_ctx(ct_ctx, max(max_ctx, 1L))
}
ct_n_ctx_cols <- sapply(ct_order, function(ct) sum(grepl("^ctx_padded_", names(ct_ordered_ctx[[ct]]))))
names(ct_n_ctx_cols) <- ct_order
cat("  Ordered ctx cols per sheet:", paste(ct_n_ctx_cols, collapse=", "), "\n")


## Inspect table

In [ ]:
head(gl_table)


## STEP 9: Write Excel output

In [ ]:
# ── Excel helper: write a sheet with 3-row header ────────────────────────
write_sheet <- function(wb, sheet_name, data, super_groups, sub_groups, col_names) {
  addWorksheet(wb, sheet_name)
  writeData(wb, sheet_name, x = t(super_groups), startRow = 1, colNames = FALSE)
  writeData(wb, sheet_name, x = t(sub_groups),   startRow = 2, colNames = FALSE)
  writeData(wb, sheet_name, x = t(col_names),    startRow = 3, colNames = FALSE)
  writeData(wb, sheet_name, x = data,            startRow = 4, colNames = FALSE)
  dark_blue  <- createStyle(fgFill="#1F4E79", fontColour="#FFFFFF", textDecoration="bold",
                             halign="center", wrapText=TRUE, fontSize=11, fontName="Arial")
  med_blue   <- createStyle(fgFill="#2E75B6", fontColour="#FFFFFF", textDecoration="bold",
                             halign="center", wrapText=TRUE, fontSize=10, fontName="Arial")
  light_blue <- createStyle(fgFill="#9DC3E6", fontColour="#000000", textDecoration="bold",
                             halign="center", wrapText=TRUE, fontSize=10, fontName="Arial")
  ncols <- length(col_names)
  addStyle(wb, sheet_name, dark_blue,  rows=1, cols=1:ncols, gridExpand=TRUE)
  addStyle(wb, sheet_name, med_blue,   rows=2, cols=1:ncols, gridExpand=TRUE)
  addStyle(wb, sheet_name, light_blue, rows=3, cols=1:ncols, gridExpand=TRUE)
  freezePane(wb, sheet_name, firstActiveRow=4, firstActiveCol=6)
}

# ── Helper: build gene-summary header vectors ─────────────────────────
build_gene_headers <- function(n_ctx_cols, n_non_linear = 8) {
  ctx_names <- rep("Ordered contexts (confidence level , n datasets)", n_ctx_cols)
  gene_sub <- c(
    rep("Gene", 11),
    rep("xQTLs variants prioritization", 5),
    rep("xQTLs contexts associated", 2 + n_ctx_cols),
    rep("All datasets associated to this variant", 3),
    rep("trans xQTL effect", 2),
    rep("Multi-genes targets  (mvSuSiE 95% CS)", 2),
    rep("Non linear effects", n_non_linear)
  )
  gene_super <- rep("xQTLs summary per gene", length(gene_sub))
  base_nl <- c("QR contexts", "Top QR context", "Top QR classification",
               "Top QR adjusted P-value (bonferroni)")
  extra_nl <- switch(as.character(n_non_linear),
    "8" = c("Interactions with APOE4", "min p-value for APOE4 interaction",
            "Interactions with Sex (M)", "min p-value for Sex interaction"),
    "6" = c("Interactions with APOE4", "min p-value for APOE4 interaction"),
    character(0)
  )
  gene_c <- c(
    "xQTL target gene", "gene ID", "tss", "tes",
    "Distance from TSS", "Distance from TES",
    "Maximum TWAS zscore", "Maximum TWAS context",
    "TWAS significant", "MR significant", "cTWAS significant",
    "Maximum inclusion score", "Variant rank", "Context", "effect",
    "# variants within the loci",
    "# molecular contexts (datasets)", "max confidence level",
    ctx_names,
    "datasets", "Methods", "Credible set coverage",
    "# trans genes", "# trans contexts",
    "# genes", "genes",
    base_nl, extra_nl
  )
  list(super = gene_super, sub = gene_sub, cols = gene_c)
}

# ── Helper: build QTL column headers ──────────────────────────────────
build_ct_headers <- function(ct, qt_list, wide_list) {
  ct_label <- ct_display_label[[ct]]
  sup <- c(); sub <- c(); cols <- c()
  for (qt in qt_list) {
    block_name <- paste0(ct, ".", qt)
    if (block_name %in% names(wide_list)) {
      bcols <- names(wide_list[[block_name]])
      bcols <- bcols[!bcols %in% c("locus_index", "variant_ID", "gene_name")]
      dcols <- sub(paste0("^", ct, "\\.", qt, "\\."), "", bcols)
      n     <- length(dcols)
      sup   <- c(sup,  rep(ct_label, n))
      sub   <- c(sub,  rep(qt, n))
      cols  <- c(cols, dcols)
    }
  }
  list(super = sup, sub = sub, cols = cols)
}

# ── Helper: assemble sheet data with positional column mapping ─────────
build_sheet_data <- function(joined, locus_header, g_cols, c_cols, q_cols) {
  joined     <- as.data.frame(joined, stringsAsFactors = FALSE)
  locus_part <- as.data.frame(lapply(locus_col_map, function(col)
    if (col %in% names(joined)) joined[[col]] else rep(NA_character_, nrow(joined))),
    stringsAsFactors = FALSE)
  gene_part  <- as.data.frame(joined[g_cols], stringsAsFactors = FALSE)
  ctx_part   <- if (length(c_cols) > 0) as.data.frame(joined[c_cols], stringsAsFactors = FALSE) else NULL
  qtl_part   <- if (length(q_cols) > 0) as.data.frame(joined[q_cols], stringsAsFactors = FALSE) else NULL
  do.call(cbind, Filter(Negate(is.null), list(locus_part, gene_part, ctx_part, qtl_part)))
}

# ── Locus/variant header vectors (25 cols) ─────────────────────────────
locus_super <- rep("Unified AD associated locus", 25)
locus_sub   <- c(rep(NA, 10), rep("AD variants priorization", 15))
locus_cols  <- c(
  "locus index", "ADlocus", "ADlocusID", "# variants in this locus",
  "# with inclusion score >01", "Variant ID", "Rsid", "Chr", "Pos",
  "Effect allele", "cV2F score", "maximum inclusion score",
  "maximum inclusion score method", "cV2F rank", "Maximum zscore",
  "Min p-value", "Significance", "log10(pval)",
  "GWAS associated to this variant",
  "GWAS Sources", "GWAS Sources", "GWAS Sources", "GWAS Sources",
  "Detected methods", "Credible set coverage (SuSiE)"
)
locus_col_map <- c(
  "locus_index", "ADlocus", "ADlocusID",
  "# variants in this locus", "# with inclusion score >01",
  "variant_ID", "Rsid", "Chr", "Pos", "Effect allele",
  "cV2F score", "maximum inclusion score",
  "maximum inclusion score method", "cV2F rank", "Maximum zscore",
  "Min p-value", "Significance", "log10(pval)",
  "GWAS associated to this variant",
  "GWAS Sources", "GWAS Sources.1", "GWAS Sources.2", "GWAS Sources.3",
  "Detected methods", "Credible set coverage (SuSiE)"
)
lvc_new_cols <- c("locus_index", "variant_ID",
                  setdiff(names(locus_variant_cols), names(gl_table)))

# ── Write workbook ──────────────────────────────────────────────────────
if (exists("wb")) rm(wb)
wb <- createWorkbook()
cat("Writing Excel output...\n")

# Gene Locus table sheet
gl_gene_h    <- build_gene_headers(n_ctx_cols = MAX_CTX_GENE_LOCUS, n_non_linear = 8)
gl_ct_h_list <- lapply(ct_order, function(ct) build_ct_headers(ct, ct_qtl_order[[ct]], gl_wide_list))
all_super_gl <- c(locus_super, gl_gene_h$super, unlist(lapply(gl_ct_h_list, `[[`, "super")))
all_sub_gl   <- c(locus_sub,   gl_gene_h$sub,   unlist(lapply(gl_ct_h_list, `[[`, "sub")))
all_cols_gl  <- c(locus_cols,  gl_gene_h$cols,  unlist(lapply(gl_ct_h_list, `[[`, "cols")))

gene_cols_data <- setdiff(names(gene_summary), c("locus_index", "variant_ID", "gene_name"))
ctx_cols_data  <- grep("^ctx_padded_", names(ordered_contexts_wide_gl), value = TRUE)
gl_qtl_cols    <- unlist(lapply(names(gl_wide_list), function(bn)
  setdiff(names(gl_wide_list[[bn]]), c("locus_index", "variant_ID", "gene_name"))))

gl_joined <- locus_variant_cols %>%
  select(all_of(lvc_new_cols)) %>%
  right_join(gl_table, by = c("locus_index", "variant_ID"))
gl_data <- build_sheet_data(gl_joined, locus_cols, gene_cols_data, ctx_cols_data, gl_qtl_cols)

if (ncol(gl_data) != length(all_cols_gl))
  warning("Gene Locus col mismatch: data=", ncol(gl_data), " header=", length(all_cols_gl))
write_sheet(wb, "Gene Locus table", gl_data, all_super_gl, all_sub_gl, all_cols_gl)
cat("  Gene Locus table:", nrow(gl_data), "rows x", ncol(gl_data), "cols written\n")

# Per-cell-type sheets (driven by ct_order and ct_non_linear_cols from CONFIG)
for (ct in ct_order) {
  sheet_tbl <- ct_sheet_data[[ct]]
  n_ctx     <- ct_n_ctx_cols[[ct]]
  if (nrow(ct_ordered_ctx[[ct]]) > 0)
    sheet_tbl <- sheet_tbl %>%
      left_join(ct_ordered_ctx[[ct]], by = c("locus_index", "variant_ID", "gene_name"))

  nl_n      <- ct_non_linear_cols[[ct]]
  ct_gene_h <- build_gene_headers(n_ctx_cols = n_ctx, n_non_linear = nl_n)

  # Immune (bulk) uses sex interaction labels instead of APOE4
  if (ct == "Immune (bulk)" && nl_n == 6) {
    ct_gene_h$cols[ct_gene_h$cols == "Interactions with APOE4"]          <- "Interactions with Sex (M)"
    ct_gene_h$cols[ct_gene_h$cols == "min p-value for APOE4 interaction"] <- "min p-value for Sex interaction"
  }

  ct_h          <- build_ct_headers(ct, ct_qtl_order[[ct]], ct_wide_list)
  ct_super_all  <- c(locus_super, ct_gene_h$super, ct_h$super)
  ct_sub_all    <- c(locus_sub,   ct_gene_h$sub,   ct_h$sub)
  ct_hcols      <- c(locus_cols,  ct_gene_h$cols,  ct_h$cols)

  ct_ctx_cols_data <- grep("^ctx_padded_", names(ct_ordered_ctx[[ct]]), value = TRUE)
  ct_qtl_cols_data <- unlist(lapply(ct_qtl_order[[ct]], function(qt) {
    bn <- paste0(ct, ".", qt)
    if (bn %in% names(ct_wide_list))
      setdiff(names(ct_wide_list[[bn]]), c("locus_index", "variant_ID", "gene_name"))
  }))

  lvc_new_cols_ct <- c("locus_index", "variant_ID",
                       setdiff(names(locus_variant_cols), names(sheet_tbl)))
  ct_joined <- locus_variant_cols %>%
    select(all_of(lvc_new_cols_ct)) %>%
    right_join(sheet_tbl, by = c("locus_index", "variant_ID"))
  ct_data_to_write <- build_sheet_data(ct_joined, locus_cols, gene_cols_data,
                                       ct_ctx_cols_data, ct_qtl_cols_data)
  if (ncol(ct_data_to_write) != length(ct_hcols))
    warning("Sheet '", ct, "' col mismatch: data=", ncol(ct_data_to_write),
            " header=", length(ct_hcols))
  write_sheet(wb, ct, ct_data_to_write, ct_super_all, ct_sub_all, ct_hcols)
  cat("  Sheet '", ct, "':", nrow(ct_data_to_write), "rows x", length(ct_hcols), "cols\n")
}

saveWorkbook(wb, output_file, overwrite = TRUE)
cat("\nDone! Saved to:", output_file, "\n")
cat("Total rows in Gene Locus table:", nrow(gl_data), "\n")


## Deploy Shiny app

In [ ]:
# One-time: set credentials (get token at shinyapps.io/admin/#/tokens)
# rsconnect::setAccountInfo(name='jenny-empawi', token='<TOKEN>', secret='<SECRET>')

rsconnect::deployApp(
  appDir  = shiny_app_dir,
  appName = "AD-xQTL-Explorer",
  account = "jenny-empawi"
)
